In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('../src'))
from sklearn.model_selection import train_test_split
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from Transformer import Transformer
from PreProcessor import PreProcessor
from ModelCollection import ModelCollection
from PipelineBuilder import PipelineBuilder
from CrossValidation import CrossValidation

In [2]:
path_train, path_test = "../data/train.csv", "../data/test.csv"
data_loader = DataLoader(path_train, path_test)
train, test = data_loader.load()

In [3]:
test

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
1455,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
1456,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml
1457,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal


In [16]:
data_splitter = DataSplitter("SalePrice", test_size=0.2)
X_train, X_test, y_train, y_test = data_splitter.split(train)
X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_train.shape[0]/(X_train.shape[0]+X_test.shape[0])

((1168, 80), (292, 80), (1168,), (292,), 0.8)

In [17]:
# ML Pipeline
num_cols, cat_cols = test.select_dtypes(include='number').columns.tolist(), test.select_dtypes(include=['category', 'object']).columns.tolist()

transformer = Transformer()
preprocessor = PreProcessor(numerical_cols=num_cols, categorical_cols=cat_cols)
preprocessor = preprocessor.build()
model_collection = ModelCollection()
model = model_collection.get('OLS')
pipeline = PipelineBuilder(transformer=transformer, preprocessor=preprocessor, model=model)
pipeline = pipeline.build()
pipeline.fit(X_train, y_train)
pred = pd.Series(pipeline.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': pred.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())

# Evaluation through cross validation
n_folds = 5
cv = CrossValidation(n_folds, pipeline)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)

              y_test           pred       Diff%
count     292.000000     292.000000  292.000000
mean   178839.811644  177466.791096   11.282874
std     87730.751259   81035.952035   14.174185
min     35311.000000   21399.000000    0.010423
25%    127000.000000  120969.500000    3.371958
50%    154150.000000  156639.500000    7.386137
75%    209175.000000  213798.750000   13.010734
max    755000.000000  567532.000000  125.557500


{'mean_mse': np.float64(1323385338.0521994),
 'std_mse': np.float64(822802896.8446913)}

In [26]:
# Hyper-parameter tunning
param_grid = {
    "model__fit_intercept": [False, True]
}
search = cv.hyper_param_tune(X, y, param_grid)

In [25]:
search.scorer_

make_scorer(mean_squared_error, greater_is_better=False, response_method='predict')